# Automated Data Analysis - LLM-Driven Healthcare Analytics

This notebook demonstrates automated analysis using the knowledge-work-plugins/data patterns.

## Features:
- 🔍 **Automated Data Exploration** - Profile datasets automatically
- 📊 **Pattern Analysis** - Detect trends, correlations, and outliers
- ✅ **Data Validation** - Quality checks and recommendations
- 📝 **Auto-Generated Reports** - JSON and Markdown outputs
- 🔄 **Batch Processing** - Analyze multiple datasets at once

## Setup

In [ ]:
import sys
sys.path.append('../scripts')

from auto_analyze import AutomatedAnalyzer
import pandas as pd
import json
from pathlib import Path
from IPython.display import display, Markdown, HTML

# Initialize analyzer
analyzer = AutomatedAnalyzer(project_root='..')

print("✓ Automated Analyzer initialized")
print(f"  Data directory: {analyzer.data_raw}")
print(f"  Reports directory: {analyzer.reports_dir}")
print(f"  Loaded {len(analyzer.prompt_templates)} command templates")

## Example 1: Explore a Single Dataset

Let's explore the infectious disease data with automated profiling.

In [ ]:
# Example dataset path (adjust based on your data)
dataset_path = "../notebooks/weekly-infectious-disease-bulletin-cases.csv"

# Run automated exploration
print("Running /explore-data analysis...\n")
results = analyzer.analyze_dataset(dataset_path, analysis_type="explore")

# Display key findings
print(f"Dataset: {results['dataset']}")
print(f"Shape: {results['shape']['rows']} rows × {results['shape']['columns']} columns")
print(f"Memory: {results['memory_usage_mb']:.2f} MB\n")

print("Columns:")
for col in results['columns']:
    print(f"  - {col}")

if results.get('quality_flags'):
    print("\n⚠️  Quality Flags:")
    for flag in results['quality_flags']:
        print(f"  - {flag}")

## Example 2: Analyze Patterns and Trends

Run pattern analysis to detect correlations, outliers, and temporal trends.

In [ ]:
# Run pattern analysis
print("Running /analyze patterns...\n")
analysis_results = analyzer.analyze_dataset(dataset_path, analysis_type="analyze")

# Display correlations
if analysis_results.get('correlations', {}).get('strong_correlations'):
    print("Strong Correlations (|r| > 0.7):")
    for corr in analysis_results['correlations']['strong_correlations']:
        print(f"  {corr['col1']} ↔ {corr['col2']}: r={corr['correlation']:.3f}")
else:
    print("No strong correlations detected")

# Display outliers
if analysis_results.get('outliers'):
    print("\nOutliers Detected:")
    for col, info in analysis_results['outliers'].items():
        print(f"  {col}: {info['count']} outliers ({info['percentage']:.2f}%)")
else:
    print("\nNo significant outliers detected")

# Display temporal info
if analysis_results.get('temporal_analysis'):
    print("\nTemporal Coverage:")
    for key, value in analysis_results['temporal_analysis'].items():
        if isinstance(value, dict):
            print(f"  {key}: {value.get('min')} to {value.get('max')}")

## Example 3: Validate Data Quality

Run validation checks to ensure data quality before analysis.

In [ ]:
# Run validation
print("Running /validate checks...\n")
validation_results = analyzer.analyze_dataset(dataset_path, analysis_type="validate")

# Display validation summary
print(f"Duplicate rows: {validation_results['duplicate_rows']} ({validation_results['duplicate_percentage']:.2f}%)")

print("\nConsistency Checks:")
for check in validation_results['consistency_checks']:
    print(f"  {check}")

print("\nRecommendations:")
for rec in validation_results['recommendations'][:10]:  # Show first 10
    print(f"  {rec}")

## Example 4: Batch Processing

Analyze multiple datasets automatically with batch processing.

In [ ]:
# Batch analyze all CSVs in a directory
data_directory = "../data/1_raw/kaggle"

print(f"Running batch analysis on {data_directory}...\n")

batch_results = analyzer.batch_analyze(
    data_directory=data_directory,
    analysis_types=['explore', 'validate']
)

# Display batch summary
print(f"\n{'='*60}")
print("BATCH ANALYSIS SUMMARY")
print(f"{'='*60}")
print(f"Files processed: {len(batch_results['files_processed'])}")

for file_result in batch_results['files_processed']:
    filename = Path(file_result['file']).name
    status = file_result['results']
    
    status_icons = []
    for analysis_type, result in status.items():
        icon = "✓" if result == "completed" else "✗"
        status_icons.append(f"{icon} {analysis_type}")
    
    print(f"\n  {filename}")
    print(f"    {' | '.join(status_icons)}")

print(f"\n{'='*60}")

## Example 5: View Generated Reports

Display the auto-generated markdown reports.

In [ ]:
# List recent reports
reports_dir = Path("../reports")
recent_reports = sorted(reports_dir.glob("*.md"), key=lambda x: x.stat().st_mtime, reverse=True)

print(f"Recent reports ({len(recent_reports)}):")
for i, report in enumerate(recent_reports[:5], 1):
    print(f"  {i}. {report.name}")

# Display the most recent report
if recent_reports:
    print(f"\nDisplaying: {recent_reports[0].name}\n")
    with open(recent_reports[0], 'r') as f:
        report_content = f.read()
    display(Markdown(report_content))

## Example 6: Custom Analysis Workflow

Build a custom analysis pipeline combining multiple analyses.

In [ ]:
def comprehensive_analysis(dataset_path: str):
    """
    Run a comprehensive analysis pipeline:
    1. Explore data
    2. Validate quality
    3. Analyze patterns
    4. Generate insights
    """
    print(f"📊 Comprehensive Analysis Pipeline")
    print(f"Dataset: {Path(dataset_path).name}\n")
    
    # Step 1: Explore
    print("[1/3] Exploring data...")
    explore_results = analyzer.analyze_dataset(dataset_path, "explore")
    print(f"  ✓ Found {explore_results['shape']['rows']} rows, {explore_results['shape']['columns']} columns")
    
    # Step 2: Validate
    print("\n[2/3] Validating quality...")
    validate_results = analyzer.analyze_dataset(dataset_path, "validate")
    print(f"  ✓ Found {validate_results['duplicate_rows']} duplicates")
    print(f"  ✓ Generated {len(validate_results['recommendations'])} recommendations")
    
    # Step 3: Analyze patterns
    print("\n[3/3] Analyzing patterns...")
    analyze_results = analyzer.analyze_dataset(dataset_path, "analyze")
    print(f"  ✓ Detected correlations and outliers")
    
    # Generate insights summary
    insights = []
    
    if explore_results.get('quality_flags'):
        insights.append(f"⚠️  {len(explore_results['quality_flags'])} data quality issues")
    
    if validate_results['duplicate_percentage'] > 5:
        insights.append(f"⚠️  High duplicate rate: {validate_results['duplicate_percentage']:.1f}%")
    
    if analyze_results.get('correlations', {}).get('strong_correlations'):
        n_corr = len(analyze_results['correlations']['strong_correlations'])
        insights.append(f"✓ Found {n_corr} strong correlations")
    
    if analyze_results.get('outliers'):
        n_outliers = len(analyze_results['outliers'])
        insights.append(f"✓ Detected outliers in {n_outliers} columns")
    
    print("\n" + "="*60)
    print("KEY INSIGHTS:")
    print("="*60)
    for insight in insights:
        print(f"  {insight}")
    print("="*60)
    
    return {
        'explore': explore_results,
        'validate': validate_results,
        'analyze': analyze_results,
        'insights': insights
    }

# Run comprehensive analysis
if Path(dataset_path).exists():
    results = comprehensive_analysis(dataset_path)
else:
    print(f"Dataset not found: {dataset_path}")

## Automated Scheduling

To run analyses automatically on a schedule, use the command-line tools:

```bash
# Single dataset analysis
python scripts/auto_analyze.py --dataset data/1_raw/kaggle/dataset.csv --type explore

# Batch processing
python scripts/auto_analyze.py --batch data/1_raw/kaggle --batch-types explore validate

# Scheduled analysis (using config)
python scripts/run_scheduled_analysis.py
```

Configure schedules in `config/auto_analysis.yml`